# Case 600 VDI factorial pass analysis

Post-processing only: this notebook reads the three existing factorial CSV outputs and does not import or call the VDI simulator. It characterises **the two factorial combinations satisfying both BESTEST annual-load bounds** without treating them as correct, validated, or optimal configurations.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
if HERE.name != '2_vdi': HERE = HERE/'2_validation/_BESTEST/2_vdi'
OUT = HERE/'results/case600_vdi_crosswalk'
modelica = pd.read_csv(OUT/'factorial_mwh_summary_modelica.csv')
native = pd.read_csv(OUT/'factorial_mwh_summary_native_vdi.csv')
paired = pd.read_csv(OUT/'factorial_mwh_paired_solar_comparison.csv')
assert len(modelica) == len(native) == len(paired) == 64
assert set(modelica.factorial_id) == set(native.factorial_id) == set(paired.factorial_id)

## Passing native-VDI-solar cases

In [2]:
factor_columns = ['topology','source_allocation','envelope','inside_h','longwave','internal_gain']
passing = native.loc[native.both_in_range].copy().sort_values('factorial_id')
assert passing.factorial_id.tolist() == ['F36','F52']
assert passing.solar_group.eq('native_vdi').all()
assert (passing.D_MWh_norm == 0).all()
compact = passing[['factorial_id',*factor_columns,'heating_MWh','cooling_MWh','D_MWh_norm']].copy()
compact['solar_pipeline'] = 'native VDI'
compact.to_csv(OUT/'factorial_passing_case_features.csv', index=False)
display(compact.round(4))

,factorial_id,topology,source_allocation,envelope,inside_h,longwave,internal_gain,heating_MWh,cooling_MWh,D_MWh_norm,solar_pipeline
0,F36,no-IW,VDI-area,layer-controlled,8.0,harmonised,Modelica-split,4.8234,5.6788,0.0,native VDI
1,F52,no-IW,Modelica-fractions,layer-controlled,8.0,harmonised,Modelica-split,4.8161,5.7484,0.0,native VDI


## Physical-feature comparison

The labels below describe the recorded factor settings. A shared feature is only common to these two cases; it is not thereby identified as the cause of passing.

In [3]:
labels = {
 'topology':'IW topology', 'source_allocation':'Solar/source allocation',
 'envelope':'Envelope convention', 'inside_h':'Inside heat-transfer coefficient',
 'longwave':'Exterior long-wave treatment', 'internal_gain':'Internal-gain allocation'}
physical = passing.set_index('factorial_id')
rows = []
for key, label in labels.items():
    a, b = physical.at['F36',key], physical.at['F52',key]
    if key == 'inside_h': a, b = f'{a:g} W/(m²·K)', f'{b:g} W/(m²·K)'
    rows.append({'Feature':label,'F36':a,'F52':b,'Relationship':'shared' if a == b else 'different'})
rows += [
 {'Feature':'Solar pipeline','F36':'native VDI','F52':'native VDI','Relationship':'shared'},
 {'Feature':'Heating (MWh)','F36':physical.at['F36','heating_MWh'],'F52':physical.at['F52','heating_MWh'],'Relationship':''},
 {'Feature':'Cooling (MWh)','F36':physical.at['F36','cooling_MWh'],'F52':physical.at['F52','cooling_MWh'],'Relationship':''}]
feature_comparison = pd.DataFrame(rows)
display(feature_comparison)

,Feature,F36,F52,Relationship
0,IW topology,no-IW,no-IW,shared
1,Solar/source allocation,VDI-area,Modelica-fractions,different
2,Envelope convention,layer-controlled,layer-controlled,shared
3,Inside heat-transfer coefficient,8 W/(m²·K),8 W/(m²·K),shared
4,Exterior long-wave treatment,harmonised,harmonised,shared
5,Internal-gain allocation,Modelica-split,Modelica-split,shared
6,Solar pipeline,native VDI,native VDI,shared
7,Heating (MWh),4.823417,4.81607,
8,Cooling (MWh),5.678802,5.748399,


In [4]:
classified = feature_comparison[feature_comparison.Relationship.isin(['shared','different'])]
shared = classified.loc[classified.Relationship.eq('shared')]
different = classified.loc[classified.Relationship.eq('different')]
print('Shared features of both passing cases:')
for _, r in shared.iterrows(): print(f"- {r['Feature']}: {r['F36']}")
print('\nFeatures that differ:')
for _, r in different.iterrows(): print(f"- {r['Feature']}: F36 = {r['F36']}; F52 = {r['F52']}")

Shared features of both passing cases:
- IW topology: no-IW
- Envelope convention: layer-controlled
- Inside heat-transfer coefficient: 8 W/(m²·K)
- Exterior long-wave treatment: harmonised
- Internal-gain allocation: Modelica-split
- Solar pipeline: native VDI

Features that differ:
- Solar/source allocation: F36 = VDI-area; F52 = Modelica-fractions


## Same physical configurations under Modelica solar

In [5]:
ids = ['F36','F52']
m = modelica.set_index('factorial_id').loc[ids]
n = native.set_index('factorial_id').loc[ids]
p = paired.set_index('factorial_id').loc[ids]
solar_comparison = pd.DataFrame({
 'factorial_id':ids,
 'Modelica-solar heating_MWh':m.heating_MWh.to_numpy(),
 'Modelica-solar cooling_MWh':m.cooling_MWh.to_numpy(),
 'native-solar heating_MWh':n.heating_MWh.to_numpy(),
 'native-solar cooling_MWh':n.cooling_MWh.to_numpy(),
 'delta_heating_MWh':p.delta_heating_MWh.to_numpy(),
 'delta_cooling_MWh':p.delta_cooling_MWh.to_numpy()})
assert np.allclose(solar_comparison['native-solar heating_MWh']-solar_comparison['Modelica-solar heating_MWh'], solar_comparison.delta_heating_MWh)
assert np.allclose(solar_comparison['native-solar cooling_MWh']-solar_comparison['Modelica-solar cooling_MWh'], solar_comparison.delta_cooling_MWh)
assert not m.both_in_range.any() and n.both_in_range.all()
display(solar_comparison.round(4))

,factorial_id,Modelica-solar heating_MWh,Modelica-solar cooling_MWh,native-solar heating_MWh,native-solar cooling_MWh,delta_heating_MWh,delta_cooling_MWh
0,F36,5.0615,5.1092,4.8234,5.6788,-0.2381,0.5696
1,F52,5.0549,5.1816,4.8161,5.7484,-0.2389,0.5668


## Compact interpretation

In [6]:
shared_physical = shared.loc[shared.Feature.ne('Solar pipeline')]
print('1. Shared physical settings: ' + '; '.join(f"{r.Feature} = {r.F36}" for _,r in shared_physical.iterrows()) + '.')
print('2. Distinguishing factor(s): ' + '; '.join(f"{r.Feature} (F36 = {r.F36}, F52 = {r.F52})" for _,r in different.iterrows()) + '.')
for _, r in solar_comparison.iterrows():
    print(f"3. {r.factorial_id}: native VDI solar changes heating by {r.delta_heating_MWh:+.4f} MWh and cooling by {r.delta_cooling_MWh:+.4f} MWh relative to Modelica solar.")

1. Shared physical settings: IW topology = no-IW; Envelope convention = layer-controlled; Inside heat-transfer coefficient = 8 W/(m²·K); Exterior long-wave treatment = harmonised; Internal-gain allocation = Modelica-split.
2. Distinguishing factor(s): Solar/source allocation (F36 = VDI-area, F52 = Modelica-fractions).
3. F36: native VDI solar changes heating by -0.2381 MWh and cooling by +0.5696 MWh relative to Modelica solar.
3. F52: native VDI solar changes heating by -0.2389 MWh and cooling by +0.5668 MWh relative to Modelica solar.


This notebook performed CSV post-processing only. No simulation package, crosswalk engine, or simulation function was imported or invoked.